In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [2]:
base_dir = '/data/aman_singh/acuuracy_check'
os.chdir(base_dir)

In [3]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [4]:
dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


### Helper functions

In [5]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

realignment_df = realignment_df[
    realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'ALL'])]


def realign_pskus(data, column):
    realignment_data = realignment_df.copy()
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

In [7]:

import glob
import re
import os

def read_forecast_file(filepath):
    
    # Read file
    df = pd.read_csv(filepath)
    
    # Extract starting month from filename
    # Example: "..._Sep 2026_to_Dec 2026_..."
    match = re.search(r'forecast_([A-Za-z]{3})\s+2026_to_', os.path.basename(filepath))
    
    if not match:
        raise ValueError(f"Could not identify starting month from: {filepath}")
    
    month = match.group(1)
    
    # Forecast column corresponding to starting month
    forecast_col = f"{month} - forecast"
    
    if forecast_col not in df.columns:
        raise ValueError(
            f"Expected column '{forecast_col}' not found in {filepath}. "
            f"Available columns: {df.columns.tolist()}"
        )
    
    # Keep item_id + required month's forecast
    df = df[['facility_name','item_id', forecast_col]].copy()
    df['chain_name'] = 'Blinkit'
    
    # Rename columns
    df.rename(
        columns={
            'item_id': 'item_code',
            forecast_col: 'forecast_quantity'
        },
        inplace=True
    )
    
    # Set date as month-end
    df['date'] = pd.to_datetime(f'2026-{pd.to_datetime(month, format="%b").month:02d}-01') \
                 + pd.offsets.MonthEnd(0)
    
    return df


# Path
path = '/data/aman_singh/acuuracy_check/'

# Get all forecast files
files = glob.glob(f'{path}Marico Ltd._forecast_*_blinkit.csv')

# Read and concatenate
blinkit_forecast = pd.concat(
    [read_forecast_file(file) for file in files],
    ignore_index=True
)

blinkit_forecast

,facility_name,item_code,forecast_quantity,chain_name,date
0,Super Store Lucknow L4 - Warehouse,10231034,0,Blinkit,2026-05-31
1,Super Store Lucknow L4 - Warehouse,10044322,2400,Blinkit,2026-05-31
2,Super Store Lucknow L4 - Warehouse,10196027,61,Blinkit,2026-05-31
3,Super Store Lucknow L4 - Warehouse,10265690,39,Blinkit,2026-05-31
4,Super Store Lucknow L4 - Warehouse,10010052,828,Blinkit,2026-05-31
...,...,...,...,...,...
32962,Visakhapatnam V1 - Feeder Warehouse,10180467,10,Blinkit,2026-09-30
32963,Visakhapatnam V1 - Feeder Warehouse,10203182,432,Blinkit,2026-09-30
32964,Visakhapatnam V1 - Feeder Warehouse,10000362,594,Blinkit,2026-09-30
32965,Visakhapatnam V1 - Feeder Warehouse,10004412,1807,Blinkit,2026-09-30


In [9]:
blinkit_forecast = blinkit_forecast.groupby(['facility_name', 'item_code', 'chain_name', 'date'], as_index=False)['forecast_quantity'].sum()
blinkit_forecast

,facility_name,item_code,chain_name,date,forecast_quantity
0,Ahmedabad A2 - Feeder Warehouse,10000059,Blinkit,2026-04-30,313
1,Ahmedabad A2 - Feeder Warehouse,10000059,Blinkit,2026-05-31,341
2,Ahmedabad A2 - Feeder Warehouse,10000059,Blinkit,2026-06-30,199
3,Ahmedabad A2 - Feeder Warehouse,10000059,Blinkit,2026-07-31,249
4,Ahmedabad A2 - Feeder Warehouse,10000059,Blinkit,2026-08-31,189
...,...,...,...,...,...
32962,Visakhapatnam V1 - Feeder Warehouse,10302659,Blinkit,2026-06-30,2
32963,Visakhapatnam V1 - Feeder Warehouse,10302659,Blinkit,2026-07-31,13
32964,Visakhapatnam V1 - Feeder Warehouse,10302659,Blinkit,2026-08-31,78
32965,Visakhapatnam V1 - Feeder Warehouse,10302659,Blinkit,2026-09-30,196


In [185]:
blinkit_forecast_sep = pd.read_csv('/data/aman_singh/acuuracy_check/Marico Ltd._forecast_Sep 2026_to_Dec 2026_sep_blinkit.csv')

blinkit_forecast_sep.rename(columns = {'item_id':'item_code', 'Sep - forecast':'forecast_quantity'}, inplace = True)
blinkit_forecast_sep.drop(columns = ['Dec - forecast', 'Oct - forecast', 'Nov - forecast'],inplace = True)
blinkit_forecast_sep['date'] = '2026-09-30'
blinkit_forecast_sep['date'] = pd.to_datetime(blinkit_forecast_sep['date'])
blinkit_forecast_sep

,facility_id,facility_name,item_code,item_name,manufacturer_id,manufacturer_name,vendor_id,vendor_name,forecast_quantity,date
0,3201,Hyderabad H3 - Feeder Warehouse,10232354,Saffola Cold Pressed Mustard Oil(Bottle)1 ltr ...,1385,Marico Ltd.,5271.0,Marico Ltd,169,2026-09-30
1,3201,Hyderabad H3 - Feeder Warehouse,10286663,"Saffola Hi- Protein Oats(Packet, Pouch)1 kg - ...",1385,Marico Ltd.,5271.0,Marico Ltd,169,2026-09-30
2,3201,Hyderabad H3 - Feeder Warehouse,10044264,Parachute Pure Coconut Hair Oil 600 ml(Pack)60...,1385,Marico Ltd.,5271.0,Marico Ltd,2145,2026-09-30
3,3201,Hyderabad H3 - Feeder Warehouse,10051713,Saffola Total Rice Bran & Sunflower Blended Co...,1385,Marico Ltd.,5271.0,Marico Ltd,644,2026-09-30
4,3201,Hyderabad H3 - Feeder Warehouse,10016872,Set Wet Ultimate Hold Hair Gel(Pack)250 ml - R...,1385,Marico Ltd.,5271.0,Marico Ltd,138,2026-09-30
...,...,...,...,...,...,...,...,...,...,...
6096,2670,Visakhapatnam V1 - Feeder Warehouse,10180467,Parachute Advansed Coconut Enriched Almond Hai...,1385,Marico Ltd.,5271.0,Marico Ltd,10,2026-09-30
6097,2670,Visakhapatnam V1 - Feeder Warehouse,10203182,Parachute Advansed Aloe Vera Enriched Coconut ...,1385,Marico Ltd.,5271.0,Marico Ltd,432,2026-09-30
6098,2670,Visakhapatnam V1 - Feeder Warehouse,10000362,Saffola Tasty + Refined Rice Bran & Corn Blend...,1385,Marico Ltd.,5271.0,Marico Ltd,594,2026-09-30
6099,2670,Visakhapatnam V1 - Feeder Warehouse,10004412,Saffola Classic-Masala Oats(Pack)38 gm - Rs 20.0,1385,Marico Ltd.,5271.0,Marico Ltd,1807,2026-09-30


In [186]:
# blinkit_forecast_may = pd.read_csv('/data/aman_singh/acuuracy_check/Marico Ltd._forecast_May 2026_to_Aug 2026.csv')

# blinkit_forecast_may.rename(columns = {'item_id':'item_code', 'May - forecast':'forecast_quantity'}, inplace = True)
# blinkit_forecast_may.drop(columns = ['Aug - forecast', 'Jun - forecast', 'Jul - forecast'],inplace = True)
# blinkit_forecast_may['date'] = '2026-05-31'
# blinkit_forecast_may['date'] = pd.to_datetime(blinkit_forecast_may['date'])
# blinkit_forecast_may

In [187]:
# blinkit_forecast_june = pd.read_csv('/data/aman_singh/acuuracy_check/Marico Ltd._forecast_Jun 2026_to_Sep 2026.csv')

# blinkit_forecast_june.rename(columns = {'item_id':'item_code', 'Jun - forecast':'forecast_quantity'}, inplace = True)
# blinkit_forecast_june.drop(columns = ['Aug - forecast', 'Sep - forecast', 'Jul - forecast'],inplace = True)
# blinkit_forecast_june['date'] = '2026-06-30'
# blinkit_forecast_june['date'] = pd.to_datetime(blinkit_forecast_june['date'])
# blinkit_forecast_june

In [188]:
#blinkit_unpivoted = pd.concat([blinkit_forecast_apr,blinkit_forecast_may,blinkit_forecast_june])
blinkit_unpivoted = blinkit_forecast_apr.copy()
blinkit_unpivoted['chain_name'] = 'Blinkit'
blinkit_unpivoted

,facility_id,facility_name,item_code,item_name,manufacturer_id,manufacturer_name,vendor_id,vendor_name,forecast_quantity,date,chain_name
0,3201,Hyderabad H3 - Feeder Warehouse,10232354,Saffola Cold Pressed Mustard Oil(Bottle)1 ltr ...,1385,Marico Ltd.,5271.0,Marico Ltd,169,2026-09-30,Blinkit
1,3201,Hyderabad H3 - Feeder Warehouse,10286663,"Saffola Hi- Protein Oats(Packet, Pouch)1 kg - ...",1385,Marico Ltd.,5271.0,Marico Ltd,169,2026-09-30,Blinkit
2,3201,Hyderabad H3 - Feeder Warehouse,10044264,Parachute Pure Coconut Hair Oil 600 ml(Pack)60...,1385,Marico Ltd.,5271.0,Marico Ltd,2145,2026-09-30,Blinkit
3,3201,Hyderabad H3 - Feeder Warehouse,10051713,Saffola Total Rice Bran & Sunflower Blended Co...,1385,Marico Ltd.,5271.0,Marico Ltd,644,2026-09-30,Blinkit
4,3201,Hyderabad H3 - Feeder Warehouse,10016872,Set Wet Ultimate Hold Hair Gel(Pack)250 ml - R...,1385,Marico Ltd.,5271.0,Marico Ltd,138,2026-09-30,Blinkit
...,...,...,...,...,...,...,...,...,...,...,...
6096,2670,Visakhapatnam V1 - Feeder Warehouse,10180467,Parachute Advansed Coconut Enriched Almond Hai...,1385,Marico Ltd.,5271.0,Marico Ltd,10,2026-09-30,Blinkit
6097,2670,Visakhapatnam V1 - Feeder Warehouse,10203182,Parachute Advansed Aloe Vera Enriched Coconut ...,1385,Marico Ltd.,5271.0,Marico Ltd,432,2026-09-30,Blinkit
6098,2670,Visakhapatnam V1 - Feeder Warehouse,10000362,Saffola Tasty + Refined Rice Bran & Corn Blend...,1385,Marico Ltd.,5271.0,Marico Ltd,594,2026-09-30,Blinkit
6099,2670,Visakhapatnam V1 - Feeder Warehouse,10004412,Saffola Classic-Masala Oats(Pack)38 gm - Rs 20.0,1385,Marico Ltd.,5271.0,Marico Ltd,1807,2026-09-30,Blinkit


In [189]:
swiggy_forecast_apr = pd.read_excel('/data/aman_singh/acuuracy_check/MARICO LIMITED_sep_swiggy.xlsx')
swiggy_forecast_apr.columns = swiggy_forecast_apr.columns.str.lower()
swiggy_forecast_apr.rename(columns = {'wh_name':'facility_name', 'sep_buy_qty':'forecast_quantity'}, inplace = True)
swiggy_forecast_apr.drop(columns = ['nov_buy_qty', 'oct_buy_qty'],inplace = True)
swiggy_forecast_apr['date'] = '2026-09-30'
swiggy_forecast_apr['date'] = pd.to_datetime(swiggy_forecast_apr['date'])
swiggy_forecast_apr

,item_code,sku_name,company,brand,city,facility_name,forecast_quantity,date
0,5903,Beardo Ultra Glow Body Wash|Brightens Skin Ton...,MARICO LIMITED,Beardo,HYDERABAD,HYD IM4,48,2026-09-30
1,5988,Beardo Godfather Perfume,MARICO LIMITED,Beardo,MUMBAI,MUM IM3,2,2026-09-30
2,6231,Saffola Masala Oats Veggie Twist,MARICO LIMITED,Saffola,BANGALORE,BLR IM1,60,2026-09-30
3,6232,Saffola Masala Oats Peppy Tomato,MARICO LIMITED,Saffola,CHENNAI,CHE AMB IM2,30,2026-09-30
4,6381,Saffola Tasty Refined Cooking Oil Blend Of Cor...,MARICO LIMITED,Saffola,KOLKATA,KOLKATA ECOM,0,2026-09-30
...,...,...,...,...,...,...,...,...
10662,991861,Kaya Sea Salt Exfoliating Shower Gel,MARICO LIMITED,Kaya,NOIDA,NOI IM1,24,2026-09-30
10663,999977,Just Herbs Serum Foundation For Face Makeup Wi...,MARICO LIMITED,Just Herbs,BANGALORE,BLR IM1,4,2026-09-30
10664,990631,"Beardo Beard & Hair Growth Oil, Natural Hair O...",MARICO LIMITED,Beardo,DELHI,DLHY GGNFC9,72,2026-09-30
10665,991861,Kaya Sea Salt Exfoliating Shower Gel,MARICO LIMITED,Kaya,NAGPUR,NAG IM1,0,2026-09-30


In [190]:
# swiggy_forecast_may = pd.read_excel('/data/aman_singh/acuuracy_check/MARICO LIMITED_swiggy_may.xlsx')
# swiggy_forecast_may.columns = swiggy_forecast_may.columns.str.lower()
# swiggy_forecast_may.rename(columns = {'wh_name':'facility_name', 'may_buy_qty':'forecast_quantity'}, inplace = True)
# swiggy_forecast_may.drop(columns = ['jul_buy_qty', 'jun_buy_qty'],inplace = True)
# swiggy_forecast_may['date'] = '2026-05-31'
# swiggy_forecast_may['date'] = pd.to_datetime(swiggy_forecast_may['date'])
# swiggy_forecast_may

In [191]:
# swiggy_forecast_jun = pd.read_excel('/data/aman_singh/acuuracy_check/MARICO LIMITED_swiggy_june.xlsx')
# swiggy_forecast_jun.columns = swiggy_forecast_jun.columns.str.lower()
# swiggy_forecast_jun.rename(columns = {'wh_name':'facility_name', 'jun_buy_qty':'forecast_quantity'}, inplace = True)
# swiggy_forecast_jun.drop(columns = ['jul_buy_qty', 'aug_buy_qty'],inplace = True)
# swiggy_forecast_jun['date'] = '2026-06-30'
# swiggy_forecast_jun['date'] = pd.to_datetime(swiggy_forecast_jun['date'])
# swiggy_forecast_jun

In [192]:
# Swiggy_unpivoted = pd.concat([swiggy_forecast_apr,swiggy_forecast_may,swiggy_forecast_jun])
Swiggy_unpivoted = swiggy_forecast_apr.copy()
Swiggy_unpivoted['chain_name'] = 'Swiggy'
Swiggy_unpivoted

,item_code,sku_name,company,brand,city,facility_name,forecast_quantity,date,chain_name
0,5903,Beardo Ultra Glow Body Wash|Brightens Skin Ton...,MARICO LIMITED,Beardo,HYDERABAD,HYD IM4,48,2026-09-30,Swiggy
1,5988,Beardo Godfather Perfume,MARICO LIMITED,Beardo,MUMBAI,MUM IM3,2,2026-09-30,Swiggy
2,6231,Saffola Masala Oats Veggie Twist,MARICO LIMITED,Saffola,BANGALORE,BLR IM1,60,2026-09-30,Swiggy
3,6232,Saffola Masala Oats Peppy Tomato,MARICO LIMITED,Saffola,CHENNAI,CHE AMB IM2,30,2026-09-30,Swiggy
4,6381,Saffola Tasty Refined Cooking Oil Blend Of Cor...,MARICO LIMITED,Saffola,KOLKATA,KOLKATA ECOM,0,2026-09-30,Swiggy
...,...,...,...,...,...,...,...,...,...
10662,991861,Kaya Sea Salt Exfoliating Shower Gel,MARICO LIMITED,Kaya,NOIDA,NOI IM1,24,2026-09-30,Swiggy
10663,999977,Just Herbs Serum Foundation For Face Makeup Wi...,MARICO LIMITED,Just Herbs,BANGALORE,BLR IM1,4,2026-09-30,Swiggy
10664,990631,"Beardo Beard & Hair Growth Oil, Natural Hair O...",MARICO LIMITED,Beardo,DELHI,DLHY GGNFC9,72,2026-09-30,Swiggy
10665,991861,Kaya Sea Salt Exfoliating Shower Gel,MARICO LIMITED,Kaya,NAGPUR,NAG IM1,0,2026-09-30,Swiggy


In [193]:
Swiggy_unpivoted['brand'].unique()

array(['Beardo', 'Saffola', 'Coco Soul', 'Hair & Care',
       'Parachute Advansed', 'Mediker', 'Nihar Naturals', 'Set Wet',
       'Revive', 'Parachute', 'Bio Oil', 'Parachute Advansed Men',
       'Saffola Munchiez', 'Pure Sense', 'Kaya', 'Just Herbs', 'Livon',
       'Saffola Fittify'], dtype=object)

In [194]:
blinkit_unpivoted = blinkit_unpivoted.groupby(['chain_name','facility_name','item_code','date'])['forecast_quantity'].sum().reset_index()
swiggy_unpivoted = Swiggy_unpivoted.groupby(['chain_name','facility_name','item_code','date'])['forecast_quantity'].sum().reset_index()


In [195]:
chain_forecast_unpivoted = pd.concat([blinkit_unpivoted,swiggy_unpivoted])
# chain_forecast_unpivoted = blinkit_unpivoted.copy()
chain_forecast_unpivoted

,chain_name,facility_name,item_code,date,forecast_quantity
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-09-30,258
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000362,2026-09-30,120
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000364,2026-09-30,3467
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000365,2026-09-30,217
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000367,2026-09-30,4061
...,...,...,...,...,...
10662,Swiggy,VIZ IM1,978064,2026-09-30,24
10663,Swiggy,VIZ IM1,980876,2026-09-30,0
10664,Swiggy,VIZ IM1,987028,2026-09-30,20
10665,Swiggy,VIZ IM1,995855,2026-09-30,4


In [196]:
mapping = pd.read_excel("/data/aman_singh/mt_forecast/Daily Offtake Tracker - Jul'26.xlsb",sheet_name = 'Mapping')


In [197]:
len_before_merge = len(chain_forecast_unpivoted)
chain_forecast_unpivoted['item_code'] = chain_forecast_unpivoted['item_code'].astype(str)
mapping['asin'] = mapping['asin'].astype(str)
temp = mapping[['platform_name','asin','EAN','PSKU','UOM','Vol per unit']].drop_duplicates()

temp = temp[temp['platform_name'].isin(['Blinkit', 'Swiggy', 'Zepto'])]
#temp['platform_name'].unique()
duplicates = temp[temp.duplicated(subset="asin", keep=False)]
duplicates



,platform_name,asin,EAN,PSKU,UOM,Vol per unit


In [198]:
temp['PSKU'] = temp['PSKU'].astype(str)
temp['EAN'] = temp['EAN'].astype(str)
temp['UOM'] = temp['UOM'].astype(str)
len_before_merge = len(chain_forecast_unpivoted)
df_chk = chain_forecast_unpivoted.merge(temp,
                  left_on = ['item_code'], right_on = ['asin'], how = 'left')
assert(len_before_merge == len(df_chk))
df_chk['date'] = pd.to_datetime(df_chk['date'])

In [199]:
df_chk

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-09-30,258,Blinkit,10000059,8901088000772,718322,KL,5000.0
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000362,2026-09-30,120,Blinkit,10000362,8901088002530,718328,KL,900.0
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000364,2026-09-30,3467,Blinkit,10000364,8901088034593,718398,KL,932.0
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000365,2026-09-30,217,Blinkit,10000365,8901088034616,718400,KL,4660.0
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000367,2026-09-30,4061,Blinkit,10000367,8901088068758,718560,TO,38.0
...,...,...,...,...,...,...,...,...,...,...,...
16763,Swiggy,VIZ IM1,978064,2026-09-30,24,NaN,NaN,NaN,NaN,NaN,NaN
16764,Swiggy,VIZ IM1,980876,2026-09-30,0,Swiggy,980876,8901088737067,732941,L,340.0
16765,Swiggy,VIZ IM1,987028,2026-09-30,20,Swiggy,987028,8901088732864,732011,L,50.0
16766,Swiggy,VIZ IM1,995855,2026-09-30,4,NaN,NaN,NaN,NaN,NaN,NaN


In [200]:
duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
duplicates.isnull().sum()

chain_name              0
facility_name           0
item_code               0
date                    0
forecast_quantity       0
platform_name        4287
asin                 4287
EAN                  4287
PSKU                 4287
UOM                  4287
Vol per unit         4287
dtype: int64

In [201]:
# new_mappings = pd.read_excel('/data/aman_singh/acuuracy_check/Alternate_export_file (2).xlsx')
# new_mappings

In [202]:
# nulls = df_chk[(df_chk['PSKU'].isna()) ]#.to_csv('missing_item_to_psku_mappings.csv')#['forecast_quantity'].sum()
# nulls

In [203]:
# nulls[nulls]

In [204]:
# nulls.duplicated(subset=['item_code'], keep=False).sum()

In [205]:
# new_mappings = new_mappings[['Key Account Article Code',
#        'SKU Code']].drop_duplicates()

In [206]:
# new_mappings[new_mappings.duplicated(subset=['Key Account Article Code'], keep=False)].sort_values(by = ['Key Account Article Code'])

In [207]:
# x2 = nulls.merge(new_mappings, left_on = ['item_code'], right_on = ['Key Account Article Code'], how = 'left')

In [208]:
# x2.dtypes

In [209]:
# Swiggy_unpivoted['item_code'] = Swiggy_unpivoted['item_code'].astype(str)
# Swiggy_unpivoted[Swiggy_unpivoted['item_code'].isin(x2['item_code'].unique())].to_csv('missing_item_to_psku_mappingsswiggy.csv', index = False)

In [210]:
# x2.to_csv('missing_item_to_psku_mappings2.csv', index = False)

In [211]:
# x2.to_csv('missing_item_to_psku_mappings2.csv', index = False)

In [212]:
df_chk[(df_chk['chain_name'] == 'Swiggy')]#['forecast_quantity'].sum()

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
6101,Swiggy,AHM DELHIVERY,3,2026-09-30,576,Swiggy,3,89002940,718299,KL,100.0
6102,Swiggy,AHM DELHIVERY,102,2026-09-30,80,Swiggy,102,8901088002530,718328,KL,900.0
6103,Swiggy,AHM DELHIVERY,103,2026-09-30,30,Swiggy,103,8901088047302,718449,L,190.0
6104,Swiggy,AHM DELHIVERY,104,2026-09-30,1120,Swiggy,104,8901088068741,718491,TO,38.0
6105,Swiggy,AHM DELHIVERY,105,2026-09-30,1120,Swiggy,105,8901088068758,718560,TO,38.0
...,...,...,...,...,...,...,...,...,...,...,...
16763,Swiggy,VIZ IM1,978064,2026-09-30,24,NaN,NaN,NaN,NaN,NaN,NaN
16764,Swiggy,VIZ IM1,980876,2026-09-30,0,Swiggy,980876,8901088737067,732941,L,340.0
16765,Swiggy,VIZ IM1,987028,2026-09-30,20,Swiggy,987028,8901088732864,732011,L,50.0
16766,Swiggy,VIZ IM1,995855,2026-09-30,4,NaN,NaN,NaN,NaN,NaN,NaN


In [213]:
#df_chk = df_chk.dropna(subset = ['PSKU'])
df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False).sum()

4453

In [214]:
# duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
# duplicates.sort_values(by=['chain_name','facility_name','PSKU','date'])[:60]

In [215]:
# duplicates.sort_values(by=['chain_name','facility_name','PSKU','date']).to_csv('duplicates_swiggy2.csv')

In [216]:
# df_chk = df_chk.sort_values('forecast_quantity', ascending=False) \
#        .drop_duplicates(subset=['chain_name', 'facility_name','PSKU' , 'date'], keep='first')
# df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False).sum()

In [217]:
df_chk.shape

(16768, 11)

In [218]:
df_chk['month_date'] = df_chk['date'] + pd.offsets.MonthEnd(0)

df_chk.rename(columns = {'item_code':'platform_code', 'EAN':'eancode', 'UOM':'uom_reporting',
                         'Vol per unit':'vol_per_unit'},inplace=True)
df_chk['vol_in_lit'] = df_chk['forecast_quantity']*df_chk['vol_per_unit']/1000
df_chk['vol_in_rum'] = df_chk.apply(
    lambda x: x['vol_in_lit'] / 1000 if x['uom_reporting'] in ['KL', 'TO'] else x['vol_in_lit'],
    axis=1
)

df_chk = df_chk.groupby(['chain_name','facility_name', 'PSKU','month_date'])[['vol_in_rum','forecast_quantity']].sum().reset_index()
df_chk['PSKU'] = df_chk['PSKU'].astype(int)
df_chk.rename(columns = {'PSKU':'parent_material_code'}, inplace = True)
df_chk

,chain_name,facility_name,parent_material_code,month_date,vol_in_rum,forecast_quantity
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-09-30,7.908,1318
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,718312,2026-09-30,0.248,248
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,718322,2026-09-30,1.290,258
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,718328,2026-09-30,0.108,120
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,718341,2026-09-30,1.511,1511
...,...,...,...,...,...,...
12373,Swiggy,VIZ IM1,810674,2026-09-30,1.008,72
12374,Swiggy,VIZ IM1,810685,2026-09-30,0.008,20
12375,Swiggy,VIZ IM1,810738,2026-09-30,8.688,24
12376,Swiggy,VIZ IM1,811181,2026-09-30,0.000,0


In [219]:
df_chk.duplicated(subset=['chain_name','facility_name','parent_material_code','month_date'], keep=False).sum()

0

In [220]:
# facility_to_city_mappings_df = pd.read_excel(
#     r'/data/aman_singh/mt_forecast/City Mappings QCOM.xlsb', 
#     'Sheet1'
# )
# facility_to_city_mappings_df.columns = facility_to_city_mappings_df.columns.str.lower()
# facility_to_city_mappings_df.columns = ['chain', 'facility_name', 'city', 'customer', 'marico_depot',
#        'status', 'depot_name']
# facility_to_city_mappings_df = facility_to_city_mappings_df[
#     facility_to_city_mappings_df['customer'].notna()
# ]
# facility_to_city_mappings_df['facility_name'] = facility_to_city_mappings_df['facility_name'].str.lower()
# facility_to_city_mappings_df['city'] = facility_to_city_mappings_df['city'].str.lower()
# facility_to_city_mappings_df.head()
# customer_depot_mappings_df = pd.read_sql("""
# SELECT DISTINCT customer_code, depot_code, channel_name 
# FROM mst_customer
# WHERE company_code='MIL' AND
#     latest_record_ind=1
# ORDER BY 3, 1, 2
# """,
# prod_conn
# )
# customer_depot_mappings_df.columns = customer_depot_mappings_df.columns.str.lower()
# customer_depot_mappings_df.duplicated(subset=['customer_code']).sum()
# customer_depot_mappings_df['customer_code'] = customer_depot_mappings_df['customer_code'].astype(str)
# facility_to_city_mappings_df['customer'] = facility_to_city_mappings_df['customer'].astype(str)
# customer_depot_mappings_df.dtypes
# facility_to_city_mappings_df.dtypes
# len_before_merge = len(facility_to_city_mappings_df)
# facility_to_city_mappings_df = facility_to_city_mappings_df.merge(
#     customer_depot_mappings_df[['customer_code', 'depot_code']].drop_duplicates().rename(
#         columns={'customer_code': 'customer'}
#     ),
#     on=['customer'],
#     how='left'
# )
# assert len_before_merge == len(facility_to_city_mappings_df)
# del len_before_merge

In [221]:
# facility_to_city_mappings_df#.isnull().sum()

In [222]:
# facility_to_city_mappings_df.rename(columns = {'facility_name':'FC', 'chain':'chain_name'}, inplace = True)
# facility_to_city_mappings_df['FC'] = facility_to_city_mappings_df['FC'].str.lower()

# facility_to_city_mappings_df[facility_to_city_mappings_df.duplicated(subset = ['chain_name','FC'],keep=False)]
# facility_to_city_mappings_df = facility_to_city_mappings_df[['chain_name','FC','depot_code']].drop_duplicates()


In [223]:
facility_to_city_mappings_df = pd.read_excel(
    r'/data/aman_singh/acuuracy_check/City Mappings QCOM.xlsb', 
    'fc_depot_mappings'
)

In [224]:
facility_to_city_mappings_df.rename(columns = {'Facility Name':'FC', 'Marico Depot':'depot_code'}, inplace = True)
facility_to_city_mappings_df['FC'] = facility_to_city_mappings_df['FC'].astype(str)
facility_to_city_mappings_df['FC'] = facility_to_city_mappings_df['FC'].str.replace('\xa0', ' ', regex=True)
facility_to_city_mappings_df['FC'] = facility_to_city_mappings_df['FC'].str.lower()
facility_to_city_mappings_df

,FC,depot_code
0,farukhnagar f2 - feeder warehouse,D115
1,ahmedabad a2 - feeder warehouse,D354
2,hyderabad h3 - feeder warehouse,D530
3,lucknow l5 - feeder warehouse,D113
4,super store hyderabad h2 - warehouse,D530
...,...,...
155,farukhnagar f3 - feeder warehouse,D115
156,hyderabad h4 - feeder warehouse,D530
157,lucknow l6 - feeder warehouse,D113
158,raipur - feeder warehouse,D248


In [225]:
facility_to_city_mappings_df.tail(10)

,FC,depot_code
150,varanasi v2 - feeder warehouse,D113
151,vijayawada - feeder warehouse,D572
152,blr im4 - feeder warehouse,D673
153,ahmedabad a3 - feeder warehouse,D354
154,chennai c6 - feeder warehouse,D674
155,farukhnagar f3 - feeder warehouse,D115
156,hyderabad h4 - feeder warehouse,D530
157,lucknow l6 - feeder warehouse,D113
158,raipur - feeder warehouse,D248
159,blr im4,D673


In [226]:
df_chk.rename(columns = {'facility_name':'FC'},inplace = True)
df_chk['FC'] = df_chk['FC'].str.lower()
df_chk

,chain_name,FC,parent_material_code,month_date,vol_in_rum,forecast_quantity
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-09-30,7.908,1318
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-09-30,0.248,248
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-09-30,1.290,258
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-09-30,0.108,120
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-09-30,1.511,1511
...,...,...,...,...,...,...
12373,Swiggy,viz im1,810674,2026-09-30,1.008,72
12374,Swiggy,viz im1,810685,2026-09-30,0.008,20
12375,Swiggy,viz im1,810738,2026-09-30,8.688,24
12376,Swiggy,viz im1,811181,2026-09-30,0.000,0


In [227]:
df_chk[df_chk['chain_name'] == 'Blinkit']['forecast_quantity'].sum()

2269136

In [228]:
chain_forecast_unpivoted[chain_forecast_unpivoted['chain_name'] == 'Blinkit']['forecast_quantity'].sum()

2275315

In [229]:
xy = df_chk.copy()
xy

,chain_name,FC,parent_material_code,month_date,vol_in_rum,forecast_quantity
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-09-30,7.908,1318
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-09-30,0.248,248
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-09-30,1.290,258
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-09-30,0.108,120
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-09-30,1.511,1511
...,...,...,...,...,...,...
12373,Swiggy,viz im1,810674,2026-09-30,1.008,72
12374,Swiggy,viz im1,810685,2026-09-30,0.008,20
12375,Swiggy,viz im1,810738,2026-09-30,8.688,24
12376,Swiggy,viz im1,811181,2026-09-30,0.000,0


In [230]:
facility_to_city_mappings_df['FC'][153]#['155']

'ahmedabad a3 - feeder warehouse'

In [231]:
facility_to_city_mappings_df[facility_to_city_mappings_df['FC']== 'ahmedabad a3 - feeder warehouse']

,FC,depot_code
153,ahmedabad a3 - feeder warehouse,D354


In [232]:
df_chk = df_chk.merge(facility_to_city_mappings_df, on = ['FC'], how = 'left')
df_chk#.isnull().sum()

,chain_name,FC,parent_material_code,month_date,vol_in_rum,forecast_quantity,depot_code
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-09-30,7.908,1318,D354
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-09-30,0.248,248,D354
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-09-30,1.290,258,D354
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-09-30,0.108,120,D354
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-09-30,1.511,1511,D354
...,...,...,...,...,...,...,...
12373,Swiggy,viz im1,810674,2026-09-30,1.008,72,D572
12374,Swiggy,viz im1,810685,2026-09-30,0.008,20,D572
12375,Swiggy,viz im1,810738,2026-09-30,8.688,24,D572
12376,Swiggy,viz im1,811181,2026-09-30,0.000,0,D572


In [233]:
df_chk[df_chk['depot_code'].isna()][['FC','chain_name']].drop_duplicates()#['vol_in_rum'].sum()/df_chk['vol_in_rum'].sum()

,FC,chain_name


In [235]:
df_chk[df_chk['chain_name'] == 'Swiggy']['vol_in_rum'].sum()

26410.117412

In [236]:
df_chk

,chain_name,FC,parent_material_code,month_date,vol_in_rum,forecast_quantity,depot_code
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-09-30,7.908,1318,D354
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-09-30,0.248,248,D354
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-09-30,1.290,258,D354
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-09-30,0.108,120,D354
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-09-30,1.511,1511,D354
...,...,...,...,...,...,...,...
12373,Swiggy,viz im1,810674,2026-09-30,1.008,72,D572
12374,Swiggy,viz im1,810685,2026-09-30,0.008,20,D572
12375,Swiggy,viz im1,810738,2026-09-30,8.688,24,D572
12376,Swiggy,viz im1,811181,2026-09-30,0.000,0,D572


In [237]:
material_master_df = pd.read_sql(
    """select * from mst_material 
    where latest_record_ind=1 and company_code='MIL'""",
    prod_conn
)
material_master_df.columns = material_master_df.columns.str.lower()
assert material_master_df.duplicated(
    subset=['company_code', 'material_code']).sum() == 0
material_master_df = material_master_df.rename(columns=
    {'material_group_code': 'brand_code'})
material_master_df['material_code'] = material_master_df['material_code'].astype(np.int64)
material_master_df.duplicated(subset=['material_code', 'parent_material_code', 'brand_code']).sum()

0

In [238]:
# material_master_df[['parent_material_code', 'brand_code']].dtypes
material_master_df['parent_material_code'] = material_master_df['parent_material_code'].astype(int)
material_master_df.loc[material_master_df['parent_material_code'].isin([725930,731857]), 'brand_code'] = 'H&C_ALMND'

# offtake_df.drop(columns = ['brand_code'],inplace = True)
len_before_merge = len(df_chk)
df_chk = df_chk.merge(
    material_master_df[['parent_material_code', 'brand_code']].drop_duplicates(),
    left_on=['parent_material_code'],right_on = ['parent_material_code'],
    how='left'
)
assert len_before_merge == len(df_chk)

In [239]:
df_chk

,chain_name,FC,parent_material_code,month_date,vol_in_rum,forecast_quantity,depot_code,brand_code
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-09-30,7.908,1318,D354,SAFF GOLD
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-09-30,0.248,248,D354,PCNO(R)
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-09-30,1.290,258,D354,SAFF KO
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-09-30,0.108,120,D354,SAFF KOCO
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-09-30,1.511,1511,D354,SAFF GOLD
...,...,...,...,...,...,...,...,...
12373,Swiggy,viz im1,810674,2026-09-30,1.008,72,D572,PA_ESS_HO
12374,Swiggy,viz im1,810685,2026-09-30,0.008,20,D572,SAF-MUSLI
12375,Swiggy,viz im1,810738,2026-09-30,8.688,24,D572,PABABY_GM
12376,Swiggy,viz im1,811181,2026-09-30,0.000,0,D572,SAF_CDPRS


In [240]:
def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate



In [241]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


len_before_merge = len(df_chk)

df_chk = df_chk.rename(columns={'material_group_code': 'brand_code'}).merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(df_chk)


Credentials retrieved successfully for prod db.


In [242]:
df_chk['value'] = df_chk['vol_in_rum']*df_chk['qtr_ind_rate']/10**7
df_chk[df_chk['chain_name'] == 'Blinkit'].groupby(['month_date'])['value'].sum()

month_date
2026-09-30    23.957488
Name: value, dtype: float64

In [243]:
df_chk[(df_chk['depot_code'].isna()) & (df_chk['chain_name'] == 'Swiggy')].groupby(['month_date'])['value'].sum()

Series([], Name: value, dtype: float64)

In [246]:
df_chk.groupby(['chain_name'])['value'].sum()

chain_name
Blinkit    23.957488
Swiggy      5.429218
Name: value, dtype: float64

In [247]:
df_chk


,chain_name,FC,parent_material_code,month_date,vol_in_rum,forecast_quantity,depot_code,brand_code,qtr_ind_rate,value
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-09-30,7.908,1318,D354,SAFF GOLD,138865.260689,0.109815
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-09-30,0.248,248,D354,PCNO(R),349274.001420,0.008662
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-09-30,1.290,258,D354,SAFF KO,168827.536176,0.021779
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-09-30,0.108,120,D354,SAFF KOCO,123636.889888,0.001335
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-09-30,1.511,1511,D354,SAFF GOLD,138865.260689,0.020983
...,...,...,...,...,...,...,...,...,...,...
12373,Swiggy,viz im1,810674,2026-09-30,1.008,72,D572,PA_ESS_HO,12860.631072,0.001296
12374,Swiggy,viz im1,810685,2026-09-30,0.008,20,D572,SAF-MUSLI,315513.490535,0.000252
12375,Swiggy,viz im1,810738,2026-09-30,8.688,24,D572,PABABY_GM,366.484998,0.000318
12376,Swiggy,viz im1,811181,2026-09-30,0.000,0,D572,SAF_CDPRS,260000.000000,0.000000


In [35]:
df_chk.to_csv('blinkit_chain_forecast.csv')

In [93]:
df_chk['value'].sum()

33.600787605504706

In [248]:
chain_forecast_zepto = pd.read_csv('/data/aman_singh/acuuracy_check/Marico_Limited_projection_sep_zepto.csv')
chain_forecast_zepto.columns = chain_forecast_zepto.columns.str.lower()
chain_forecast_zepto

,month,cluster_dry,product_variant_id,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,projected_qty
0,Sept,SAS Nagar,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,2618.0
1,Sept,Pune,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,4571.0
2,Sept,NCR,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,17508.0
3,Sept,Mumbai,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,24845.0
4,Sept,Lucknow,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,3532.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
616,Nov,Bengaluru,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,820.0,GRAM,158.0,6172.0
617,Oct,Hyderabad,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,820.0,GRAM,158.0,6259.0
618,Oct,Coimbatore,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,820.0,GRAM,158.0,572.0
619,Oct,Chennai,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,820.0,GRAM,158.0,6501.0


In [249]:
chain_forecast_zepto['month'].unique()

array(['Sept', 'Nov', 'Oct'], dtype=object)

In [250]:

month_map = {
    'Nov': '2026-11-30',
    'Sept': '2026-09-30',
    'Oct': '2026-10-31'
}

# Apply mapping
chain_forecast_zepto['date'] = chain_forecast_zepto['month'].map(month_map)
chain_forecast_zepto['date'] = pd.to_datetime(chain_forecast_zepto['date'])
chain_forecast_zepto_april = chain_forecast_zepto[chain_forecast_zepto['date'] == '2026-09-30']
chain_forecast_zepto_april

,month,cluster_dry,product_variant_id,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,projected_qty,date
0,Sept,SAS Nagar,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,2618.0,2026-09-30
1,Sept,Pune,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,4571.0,2026-09-30
2,Sept,NCR,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,17508.0,2026-09-30
3,Sept,Mumbai,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,24845.0,2026-09-30
4,Sept,Lucknow,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,3532.0,2026-09-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
602,Sept,Kolkata,289ac796-827e-4070-aabd-6d727ed62bdc,"Parachute Advansed Jasmine, Non Sticky Coconut...",Hair Care,Hair Oil,Hair Oil,Marico Limited,Marico Limited,300.0,MILLILITRE,139.0,1102.0,2026-09-30
609,Sept,Hyderabad,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,820.0,GRAM,158.0,3686.0,2026-09-30
610,Sept,Coimbatore,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,820.0,GRAM,158.0,553.0,2026-09-30
611,Sept,Chennai,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,820.0,GRAM,158.0,6360.0,2026-09-30


In [43]:
chain_forecast_zepto = pd.read_csv('/data/aman_singh/acuuracy_check/Marico_Limited_projection_jun_zepto.csv')
chain_forecast_zepto.columns = chain_forecast_zepto.columns.str.lower()
chain_forecast_zepto
chain_forecast_zepto['month'].unique()

month_map = {
    'Jun': '2026-06-30',
    'May': '2026-05-31',
    'Jul': '2026-07-31'
}

# Apply mapping
chain_forecast_zepto['date'] = chain_forecast_zepto['month'].map(month_map)
chain_forecast_zepto['date'] = pd.to_datetime(chain_forecast_zepto['date'])
chain_forecast_zepto_may = chain_forecast_zepto[chain_forecast_zepto['date'].isin(['2026-05-31'])]
chain_forecast_zepto_may

,month,cluster_dry,product_variant_id,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,projected_qty,date
0,May,Jaipur,f1b1c703-46a6-43e2-8b49-43c6a193a2a2,Bio Oil Original Skincare Oil Suitable For Str...,Skincare,Body Care,Body Oil,Bio Oil,Marico Limited,60.0,MILLILITRE,550.0,20.0,2026-05-31
3,May,SAS Nagar,a74eaad2-e93b-46d3-8e6d-d11a56d38632,Parachute Advansed Baby Wipes with virgin coco...,Baby Care,Baby Wipes,Baby Wipes,Parachute,Marico Limited,1.0,PIECE,230.0,12.0,2026-05-31
8,May,Mumbai,a0841095-9583-4788-8c67-cbe41ddab0d7,Just Herbs Eyeliner | Multicolour | Waterproof,Makeup & Beauty,Eye Makeup,Eye Liner,Justherbs,Marico Limited,42.0,GRAM,599.0,60.0,2026-05-31
10,May,Chennai,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,261.0,2104.0,2026-05-31
12,May,Ahmedabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,261.0,1104.0,2026-05-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5579,May,Jaipur,f702765e-ea47-41a5-90e8-3c797e6b29fe,Saffola 25g High Protein Oats | 14g Fibre | No...,Breakfast & Sauces,Muesli & Oats,Oats,Saffola Foods,Marico Limited,400.0,GRAM,299.0,28.0,2026-05-31
5581,May,Pune,03843645-8523-4571-8c92-7b3b5d292930,Livon Style Pro Keratin Serum 10X Stronger & S...,Hair Care,Hair Serum & Polish,Hair Serum,Livon,Marico Limited,100.0,MILLILITRE,665.0,12.0,2026-05-31
5583,May,Kolkata,da954c80-83a5-43f0-8c50-2354cd689945,Just Herbs Hair Growth Oil With Rosemary And C...,Hair Care,Hair Oil,Hair Growth Oil,Justherbs,Marico Limited,100.0,MILLILITRE,345.0,8.0,2026-05-31
5586,May,Pune,d34d554d-671a-42a7-bd48-31301ccbf729,Just Herbs Pigmented Smudge & Sweat Proof Quic...,Makeup & Beauty,Face Makeup,Sindoor,Justherbs,Marico Limited,3.5,GRAM,225.0,49.0,2026-05-31


In [44]:
chain_forecast_zepto_may['projected_qty'].sum()

1280872.0

In [13]:
chain_forecast_zepto = pd.read_csv('/data/aman_singh/acuuracy_check/Marico_Limited_projection_may_zepto.csv')
chain_forecast_zepto['Month'].unique()

array(['Jul', 'Aug', 'Jun'], dtype=object)

In [15]:
chain_forecast_zepto = pd.read_csv('/data/aman_singh/acuuracy_check/Marico_Limited_projection_may_zepto.csv')
chain_forecast_zepto.columns = chain_forecast_zepto.columns.str.lower()
chain_forecast_zepto
chain_forecast_zepto['month'].unique()

month_map = {
    'Jun': '2026-06-30',
    'Aug': '2026-08-31',
    'Jul': '2026-07-31'
}

# Apply mapping
chain_forecast_zepto['date'] = chain_forecast_zepto['month'].map(month_map)
chain_forecast_zepto['date'] = pd.to_datetime(chain_forecast_zepto['date'])
chain_forecast_zepto_june = chain_forecast_zepto[chain_forecast_zepto['date'].isin(['2026-06-30'])]
chain_forecast_zepto_june

,month,cluster_dry,product_variant_id,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,projected_qty,date
4,Jun,Chennai,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,2278,2026-06-30
6,Jun,Bengaluru,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,4991,2026-06-30
8,Jun,Pune,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,1389,2026-06-30
10,Jun,Hyderabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,7467,2026-06-30
16,Jun,Ahmedabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,1069,2026-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,Jun,Kolkata,bdb2302e-6274-4b99-ad19-3339b6cba494,Saffola Muesli Kesar Crunch With Flavour Pops ...,Breakfast & Sauces,Muesli & Oats,Muesli,Saffola Foods,Marico Limited,185.0,GRAM,149.0,1064,2026-06-30
534,Jun,Kolkata,f0e0ef4b-2a02-4e13-8498-35b32f554274,Saffola Mealmaker Soya Chunks Pouch,"Atta, Rice, Oil & Dals",Dals & Pulses,Soya Chunks,Saffola Foods,Marico Limited,400.0,GRAM,111.0,950,2026-06-30
536,Jun,Mumbai,bdb2302e-6274-4b99-ad19-3339b6cba494,Saffola Muesli Kesar Crunch With Flavour Pops ...,Breakfast & Sauces,Muesli & Oats,Muesli,Saffola Foods,Marico Limited,185.0,GRAM,149.0,171,2026-06-30
541,Jun,Hyderabad,f0e0ef4b-2a02-4e13-8498-35b32f554274,Saffola Mealmaker Soya Chunks Pouch,"Atta, Rice, Oil & Dals",Dals & Pulses,Soya Chunks,Saffola Foods,Marico Limited,400.0,GRAM,111.0,1386,2026-06-30


In [251]:
# chain_forecast_zepto = pd.concat([chain_forecast_zepto_april,chain_forecast_zepto_may,chain_forecast_zepto_june])
chain_forecast_zepto = chain_forecast_zepto_april.copy()
chain_forecast_zepto

,month,cluster_dry,product_variant_id,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,projected_qty,date
0,Sept,SAS Nagar,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,2618.0,2026-09-30
1,Sept,Pune,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,4571.0,2026-09-30
2,Sept,NCR,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,17508.0,2026-09-30
3,Sept,Mumbai,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,24845.0,2026-09-30
4,Sept,Lucknow,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,3532.0,2026-09-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
602,Sept,Kolkata,289ac796-827e-4070-aabd-6d727ed62bdc,"Parachute Advansed Jasmine, Non Sticky Coconut...",Hair Care,Hair Oil,Hair Oil,Marico Limited,Marico Limited,300.0,MILLILITRE,139.0,1102.0,2026-09-30
609,Sept,Hyderabad,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,820.0,GRAM,158.0,3686.0,2026-09-30
610,Sept,Coimbatore,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,820.0,GRAM,158.0,553.0,2026-09-30
611,Sept,Chennai,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,820.0,GRAM,158.0,6360.0,2026-09-30


In [252]:
chain_forecast_zepto['chain_name'] = 'Zepto'
chain_forecast_zepto.rename(columns = {'product_variant_id':'item_code', 
                                       'projected_qty':'forecast_quantity',
                                       'cluster_dry':'city'}, inplace = True)
chain_forecast_zepto

,month,city,item_code,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,forecast_quantity,date,chain_name
0,Sept,SAS Nagar,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,2618.0,2026-09-30,Zepto
1,Sept,Pune,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,4571.0,2026-09-30,Zepto
2,Sept,NCR,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,17508.0,2026-09-30,Zepto
3,Sept,Mumbai,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,24845.0,2026-09-30,Zepto
4,Sept,Lucknow,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,850.0,GRAM,166.0,3532.0,2026-09-30,Zepto
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
602,Sept,Kolkata,289ac796-827e-4070-aabd-6d727ed62bdc,"Parachute Advansed Jasmine, Non Sticky Coconut...",Hair Care,Hair Oil,Hair Oil,Marico Limited,Marico Limited,300.0,MILLILITRE,139.0,1102.0,2026-09-30,Zepto
609,Sept,Hyderabad,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,820.0,GRAM,158.0,3686.0,2026-09-30,Zepto
610,Sept,Coimbatore,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,820.0,GRAM,158.0,553.0,2026-09-30,Zepto
611,Sept,Chennai,19823d3a-a52a-43e9-84b3-498de8bcda2f,Saffola Tasty + Refined Rice bran & Corn Oil |...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Marico Limited,Marico Limited,820.0,GRAM,158.0,6360.0,2026-09-30,Zepto


In [253]:
chain_forecast_zepto.isnull().sum()

month                0
city                 0
item_code            0
product_name         0
category_name        0
subcategory_name     0
l3_category_name     0
brand_name           0
manufacturer         0
packsize             0
unit_of_measure      0
unit_mrp             0
forecast_quantity    0
date                 0
chain_name           0
dtype: int64

In [254]:
chain_forecast_zepto = chain_forecast_zepto.groupby(['chain_name','city','item_code','date'])['forecast_quantity'].sum().reset_index()
chain_forecast_zepto

,chain_name,city,item_code,date,forecast_quantity
0,Zepto,Ahmedabad,25f79f6e-98bc-4ec1-94b5-283f1561ff94,2026-09-30,738.0
1,Zepto,Ahmedabad,292f1828-6db0-4af6-9abd-b8e9095452ae,2026-09-30,1102.0
2,Zepto,Ahmedabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,2026-09-30,1333.0
3,Zepto,Ahmedabad,589bce9e-e0bc-467d-a6d0-10838a165e43,2026-09-30,1553.0
4,Zepto,Ahmedabad,709bd327-baf4-4104-a46d-fdba2b80b99c,2026-09-30,3129.0
...,...,...,...,...,...
202,Zepto,SAS Nagar,ce9c76b1-d2df-4b96-9fc0-46a7e9b16be5,2026-09-30,1452.0
203,Zepto,SAS Nagar,e011578a-374b-406a-890d-f092005c203d,2026-09-30,8254.0
204,Zepto,SAS Nagar,e2eeab46-1109-41ae-b90b-a9c3a429e62e,2026-09-30,1302.0
205,Zepto,SAS Nagar,e6ab8174-00bd-4a14-b59f-494502928958,2026-09-30,3450.0


In [255]:
zepto_mapping = pd.read_excel('/data/aman_singh/mt_forecast/Q-com Depot-FC-City Mapping v2.0.xlsx', sheet_name = 'Zepto')
zepto_mapping = zepto_mapping[zepto_mapping['Status'] == 'Active']
zepto_mapping = zepto_mapping[['Channel','City', 'Marico Depot']].drop_duplicates()
zepto_mapping.rename(columns = {'Channel':'platform_name', 'City':'city', 'Marico Depot':'depot'}, inplace = True)
zepto_mapping['platform_name'] = zepto_mapping['platform_name'].str.lower()
zepto_mapping['city'] = zepto_mapping['city'].str.lower()
zepto_mapping

,platform_name,city,depot
0,zepto,ahmedabad,D354
2,zepto,indore,D354
3,zepto,mehsana,D354
4,zepto,rajkot,D354
5,zepto,surat,D354
...,...,...,...
110,zepto,pune,D461
111,zepto,bahadurgarh,D115
112,zepto,gurgaon,NaN
113,zepto,raipur,D465


In [257]:
xx = chain_forecast_zepto.copy()

In [258]:
chain_forecast_zepto['city'] = chain_forecast_zepto['city'].str.lower()
chain_forecast_zepto = chain_forecast_zepto.merge(zepto_mapping, on = ['city'], how = 'left')
chain_forecast_zepto

,chain_name,city,item_code,date,forecast_quantity,platform_name,depot
0,Zepto,ahmedabad,25f79f6e-98bc-4ec1-94b5-283f1561ff94,2026-09-30,738.0,zepto,D354
1,Zepto,ahmedabad,292f1828-6db0-4af6-9abd-b8e9095452ae,2026-09-30,1102.0,zepto,D354
2,Zepto,ahmedabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,2026-09-30,1333.0,zepto,D354
3,Zepto,ahmedabad,589bce9e-e0bc-467d-a6d0-10838a165e43,2026-09-30,1553.0,zepto,D354
4,Zepto,ahmedabad,709bd327-baf4-4104-a46d-fdba2b80b99c,2026-09-30,3129.0,zepto,D354
...,...,...,...,...,...,...,...
202,Zepto,sas nagar,ce9c76b1-d2df-4b96-9fc0-46a7e9b16be5,2026-09-30,1452.0,zepto,D115
203,Zepto,sas nagar,e011578a-374b-406a-890d-f092005c203d,2026-09-30,8254.0,zepto,D115
204,Zepto,sas nagar,e2eeab46-1109-41ae-b90b-a9c3a429e62e,2026-09-30,1302.0,zepto,D115
205,Zepto,sas nagar,e6ab8174-00bd-4a14-b59f-494502928958,2026-09-30,3450.0,zepto,D115


In [259]:
chain_forecast_zepto[chain_forecast_zepto['depot'].isna()]#['city'].unique()

,chain_name,city,item_code,date,forecast_quantity,platform_name,depot


In [260]:
len_before_merge = len(chain_forecast_zepto)
chain_forecast_zepto['item_code'] = chain_forecast_zepto['item_code'].astype(str)
mapping['asin'] = mapping['asin'].astype(str)
temp = mapping[['platform_name','asin','EAN','PSKU','UOM','Vol per unit']].drop_duplicates()

temp = temp[temp['platform_name'].isin(['Blinkit', 'Swiggy', 'Zepto'])]
#temp['platform_name'].unique()
duplicates = temp[temp.duplicated(subset="asin", keep=False)]
duplicates



,platform_name,asin,EAN,PSKU,UOM,Vol per unit


In [261]:
temp['PSKU'] = temp['PSKU'].astype(str)
temp['EAN'] = temp['EAN'].astype(str)
temp['UOM'] = temp['UOM'].astype(str)
len_before_merge = len(chain_forecast_zepto)
chain_forecast_zepto = chain_forecast_zepto.merge(temp,
                  left_on = ['item_code'], right_on = ['asin'], how = 'left')
assert(len_before_merge == len(chain_forecast_zepto))
chain_forecast_zepto['date'] = pd.to_datetime(chain_forecast_zepto['date'])
chain_forecast_zepto

,chain_name,city,item_code,date,forecast_quantity,platform_name_x,depot,platform_name_y,asin,EAN,PSKU,UOM,Vol per unit
0,Zepto,ahmedabad,25f79f6e-98bc-4ec1-94b5-283f1561ff94,2026-09-30,738.0,zepto,D354,Zepto,25f79f6e-98bc-4ec1-94b5-283f1561ff94,8901088150095,719085,L,280.0
1,Zepto,ahmedabad,292f1828-6db0-4af6-9abd-b8e9095452ae,2026-09-30,1102.0,zepto,D354,Zepto,292f1828-6db0-4af6-9abd-b8e9095452ae,8901088155496,718464,TO,500.0
2,Zepto,ahmedabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,2026-09-30,1333.0,zepto,D354,Zepto,29b306e4-5656-4326-ad7f-4ecd98abf3d8,8901088102872,718825,KL,600.0
3,Zepto,ahmedabad,589bce9e-e0bc-467d-a6d0-10838a165e43,2026-09-30,1553.0,zepto,D354,Zepto,589bce9e-e0bc-467d-a6d0-10838a165e43,8901088171755,719162,TO,250.0
4,Zepto,ahmedabad,709bd327-baf4-4104-a46d-fdba2b80b99c,2026-09-30,3129.0,zepto,D354,Zepto,709bd327-baf4-4104-a46d-fdba2b80b99c,8901088136945,718828,KL,300.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,Zepto,sas nagar,ce9c76b1-d2df-4b96-9fc0-46a7e9b16be5,2026-09-30,1452.0,zepto,D115,Zepto,ce9c76b1-d2df-4b96-9fc0-46a7e9b16be5,89002681,718371,L,50.0
203,Zepto,sas nagar,e011578a-374b-406a-890d-f092005c203d,2026-09-30,8254.0,zepto,D115,Zepto,e011578a-374b-406a-890d-f092005c203d,8901088068734,718559,TO,38.0
204,Zepto,sas nagar,e2eeab46-1109-41ae-b90b-a9c3a429e62e,2026-09-30,1302.0,zepto,D115,Zepto,e2eeab46-1109-41ae-b90b-a9c3a429e62e,8901088017381,718341,KL,1000.0
205,Zepto,sas nagar,e6ab8174-00bd-4a14-b59f-494502928958,2026-09-30,3450.0,zepto,D115,Zepto,e6ab8174-00bd-4a14-b59f-494502928958,8901088068758,718560,TO,38.0


In [262]:
xx = chain_forecast_zepto.copy()

In [263]:
chain_forecast_zepto.isnull().sum()

chain_name           0
city                 0
item_code            0
date                 0
forecast_quantity    0
platform_name_x      0
depot                0
platform_name_y      9
asin                 9
EAN                  9
PSKU                 9
UOM                  9
Vol per unit         9
dtype: int64

In [264]:
chain_forecast_zepto.dropna(subset = ['PSKU'],inplace = True)
chain_forecast_zepto

,chain_name,city,item_code,date,forecast_quantity,platform_name_x,depot,platform_name_y,asin,EAN,PSKU,UOM,Vol per unit
0,Zepto,ahmedabad,25f79f6e-98bc-4ec1-94b5-283f1561ff94,2026-09-30,738.0,zepto,D354,Zepto,25f79f6e-98bc-4ec1-94b5-283f1561ff94,8901088150095,719085,L,280.0
1,Zepto,ahmedabad,292f1828-6db0-4af6-9abd-b8e9095452ae,2026-09-30,1102.0,zepto,D354,Zepto,292f1828-6db0-4af6-9abd-b8e9095452ae,8901088155496,718464,TO,500.0
2,Zepto,ahmedabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,2026-09-30,1333.0,zepto,D354,Zepto,29b306e4-5656-4326-ad7f-4ecd98abf3d8,8901088102872,718825,KL,600.0
3,Zepto,ahmedabad,589bce9e-e0bc-467d-a6d0-10838a165e43,2026-09-30,1553.0,zepto,D354,Zepto,589bce9e-e0bc-467d-a6d0-10838a165e43,8901088171755,719162,TO,250.0
4,Zepto,ahmedabad,709bd327-baf4-4104-a46d-fdba2b80b99c,2026-09-30,3129.0,zepto,D354,Zepto,709bd327-baf4-4104-a46d-fdba2b80b99c,8901088136945,718828,KL,300.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,Zepto,sas nagar,ce9c76b1-d2df-4b96-9fc0-46a7e9b16be5,2026-09-30,1452.0,zepto,D115,Zepto,ce9c76b1-d2df-4b96-9fc0-46a7e9b16be5,89002681,718371,L,50.0
203,Zepto,sas nagar,e011578a-374b-406a-890d-f092005c203d,2026-09-30,8254.0,zepto,D115,Zepto,e011578a-374b-406a-890d-f092005c203d,8901088068734,718559,TO,38.0
204,Zepto,sas nagar,e2eeab46-1109-41ae-b90b-a9c3a429e62e,2026-09-30,1302.0,zepto,D115,Zepto,e2eeab46-1109-41ae-b90b-a9c3a429e62e,8901088017381,718341,KL,1000.0
205,Zepto,sas nagar,e6ab8174-00bd-4a14-b59f-494502928958,2026-09-30,3450.0,zepto,D115,Zepto,e6ab8174-00bd-4a14-b59f-494502928958,8901088068758,718560,TO,38.0


In [ ]:
# chain_forecast_zepto.duplicated(subset=['chain_name','PSKU','date'], keep=False).sum()

2960

In [265]:
chain_forecast_zepto['month_date'] = chain_forecast_zepto['date'] + pd.offsets.MonthEnd(0)

chain_forecast_zepto.rename(columns = {'item_code':'platform_code', 'EAN':'eancode', 'UOM':'uom_reporting',
                         'Vol per unit':'vol_per_unit'},inplace=True)
chain_forecast_zepto['vol_in_lit'] = chain_forecast_zepto['forecast_quantity']*chain_forecast_zepto['vol_per_unit']/1000
chain_forecast_zepto['vol_in_rum'] = chain_forecast_zepto.apply(
    lambda x: x['vol_in_lit'] / 1000 if x['uom_reporting'] in ['KL', 'TO'] else x['vol_in_lit'],
    axis=1
)

chain_forecast_zepto = chain_forecast_zepto.groupby(['chain_name','depot','PSKU','month_date'])[['vol_in_rum','forecast_quantity']].sum().reset_index()
chain_forecast_zepto['PSKU'] = chain_forecast_zepto['PSKU'].astype(int)
chain_forecast_zepto.rename(columns = {'PSKU':'parent_material_code'}, inplace = True)
chain_forecast_zepto

,chain_name,depot,parent_material_code,month_date,vol_in_rum,forecast_quantity
0,Zepto,D112,718341,2026-09-30,12.504000,12504.0
1,Zepto,D112,718371,2026-09-30,328.500000,6570.0
2,Zepto,D112,718398,2026-09-30,16.317456,17508.0
3,Zepto,D112,718464,2026-09-30,1.728500,3457.0
4,Zepto,D112,718465,2026-09-30,4.524000,3480.0
...,...,...,...,...,...,...
180,Zepto,D674,718850,2026-09-30,39.700000,397.0
181,Zepto,D674,719085,2026-09-30,535.640000,1913.0
182,Zepto,D674,721427,2026-09-30,1.535000,1535.0
183,Zepto,D674,727811,2026-09-30,16.299000,1811.0


In [266]:
chain_forecast_zepto['forecast_quantity'].sum()

823226.0

In [267]:
xx['forecast_quantity'].sum()

836401.0

In [77]:
chain_forecast_zepto.to_csv('forecast_zepto.csv')

In [268]:
df_chk

,chain_name,FC,parent_material_code,month_date,vol_in_rum,forecast_quantity,depot_code,brand_code,qtr_ind_rate,value
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-09-30,7.908,1318,D354,SAFF GOLD,138865.260689,0.109815
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-09-30,0.248,248,D354,PCNO(R),349274.001420,0.008662
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-09-30,1.290,258,D354,SAFF KO,168827.536176,0.021779
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-09-30,0.108,120,D354,SAFF KOCO,123636.889888,0.001335
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-09-30,1.511,1511,D354,SAFF GOLD,138865.260689,0.020983
...,...,...,...,...,...,...,...,...,...,...
12373,Swiggy,viz im1,810674,2026-09-30,1.008,72,D572,PA_ESS_HO,12860.631072,0.001296
12374,Swiggy,viz im1,810685,2026-09-30,0.008,20,D572,SAF-MUSLI,315513.490535,0.000252
12375,Swiggy,viz im1,810738,2026-09-30,8.688,24,D572,PABABY_GM,366.484998,0.000318
12376,Swiggy,viz im1,811181,2026-09-30,0.000,0,D572,SAF_CDPRS,260000.000000,0.000000


In [269]:
df_chk['FC'] = df_chk['FC'].str.lower()

In [ ]:
# fc_depot_mapping = {
#     'guwahati g2 - feeder warehouse': 'D236',
#     'indore i2 - feeder warehouse': 'D464',
#     'jaipur j4 - feeder warehouse': 'D314',
#     'mumbai m12 - feeder warehouse': 'D356',
#     'patna p2 - feeder warehouse': 'D233',
#     'pune p3 - feeder warehouse': 'D461',
#     'ranchi r2 - feeder warehouse': 'D234',
#     'surat s2 - feeder warehouse': 'D354',
#     'varanasi v2 - feeder warehouse': 'D113',
#     'vijayawada - feeder warehouse': 'D572',
#     'blr im4 - feeder warehouse': 'D673'
# }
# # Fill only null depot codes
# df_chk.loc[df_chk['depot_code'].isna(), 'depot_code'] = (
#     df_chk.loc[df_chk['depot_code'].isna(), 'FC']
#           .str.lower()
#           .map(fc_depot_mapping)
# )
# df_chk

,chain_name,FC,parent_material_code,month_date,vol_in_rum,forecast_quantity,depot_code,brand_code,qtr_ind_rate,value
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-09-30,7.0680,1178,D354,SAFF GOLD,138865.260689,0.098150
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-09-30,0.3420,342,D354,PCNO(R),349274.001420,0.011945
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-09-30,1.1500,230,D354,SAFF KO,168827.536176,0.019415
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-09-30,0.1026,114,D354,SAFF KOCO,123636.889888,0.001269
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-09-30,1.5370,1537,D354,SAFF GOLD,138865.260689,0.021344
...,...,...,...,...,...,...,...,...,...,...
5436,Blinkit,visakhapatnam v1 - feeder warehouse,810520,2026-09-30,0.0010,1,D572,SAF_CDPRS,260000.000000,0.000026
5437,Blinkit,visakhapatnam v1 - feeder warehouse,810673,2026-09-30,0.8400,60,D572,PA_ESS_HO,12860.631072,0.001080
5438,Blinkit,visakhapatnam v1 - feeder warehouse,810674,2026-09-30,0.3220,23,D572,PA_ESS_HO,12860.631072,0.000414
5439,Blinkit,visakhapatnam v1 - feeder warehouse,810971,2026-09-30,0.1500,5,D572,PA_ESS_HO,12860.631072,0.000193


In [270]:
df_chk = df_chk.groupby(['chain_name','depot_code','parent_material_code','month_date'])[['vol_in_rum','forecast_quantity']].sum().reset_index()
df_chk

,chain_name,depot_code,parent_material_code,month_date,vol_in_rum,forecast_quantity
0,Blinkit,D112,718288,2026-09-30,0.6420,107
1,Blinkit,D112,718312,2026-09-30,1.1100,1110
2,Blinkit,D112,718322,2026-09-30,2.8750,575
3,Blinkit,D112,718328,2026-09-30,0.3906,434
4,Blinkit,D112,718330,2026-09-30,0.3450,69
...,...,...,...,...,...,...
6493,Swiggy,D677,810673,2026-09-30,0.5040,36
6494,Swiggy,D677,810674,2026-09-30,0.5040,36
6495,Swiggy,D677,810738,2026-09-30,0.0000,0
6496,Swiggy,D677,811181,2026-09-30,0.0240,24


In [60]:
df_chk.to_csv('blinkit_chain_forecast2.csv')

In [271]:
df_chk.rename(columns = {'depot_code':'depot'},inplace = True)
final_df = pd.concat([df_chk,chain_forecast_zepto])
final_df

,chain_name,depot,parent_material_code,month_date,vol_in_rum,forecast_quantity
0,Blinkit,D112,718288,2026-09-30,0.64200,107.0
1,Blinkit,D112,718312,2026-09-30,1.11000,1110.0
2,Blinkit,D112,718322,2026-09-30,2.87500,575.0
3,Blinkit,D112,718328,2026-09-30,0.39060,434.0
4,Blinkit,D112,718330,2026-09-30,0.34500,69.0
...,...,...,...,...,...,...
180,Zepto,D674,718850,2026-09-30,39.70000,397.0
181,Zepto,D674,719085,2026-09-30,535.64000,1913.0
182,Zepto,D674,721427,2026-09-30,1.53500,1535.0
183,Zepto,D674,727811,2026-09-30,16.29900,1811.0


In [272]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)

realignment_df.columns = realignment_df.columns.str.lower()
realignment_df['channel'].unique()
realignment_df = realignment_df[
    realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'All'])]
def realign_pskus(data, column):
    realignment_data = realignment_df.copy()
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data


In [273]:
final_df.columns

Index(['chain_name', 'depot', 'parent_material_code', 'month_date',
       'vol_in_rum', 'forecast_quantity'],
      dtype='object')

In [274]:
final_df['realigned_psku'] = final_df['parent_material_code'].copy()
final_df = realign_pskus(final_df, 'realigned_psku')
base_df = final_df.copy()
final_df = final_df.groupby(
    ['chain_name', 'depot', 'realigned_psku', 'month_date'],
    as_index=False
)[[ 'vol_in_rum', 'forecast_quantity']].sum()
final_df.duplicated(
    subset=['chain_name', 'depot', 'realigned_psku', 'month_date']).sum()

0

In [ ]:
# final_df.isnull().sum()

chain_name           0
depot                0
realigned_psku       0
month_date           0
vol_in_rum           0
forecast_quantity    0
dtype: int64

In [275]:
xx = final_df.copy()

In [102]:
xx

,chain_name,depot,realigned_psku,month_date,vol_in_rum,forecast_quantity
0,Blinkit,D112,718288,2026-08-31,0.0420,7.0
1,Blinkit,D112,718312,2026-08-31,1.7480,1748.0
2,Blinkit,D112,718322,2026-08-31,2.6650,533.0
3,Blinkit,D112,718328,2026-08-31,0.2574,286.0
4,Blinkit,D112,718330,2026-08-31,0.0200,4.0
...,...,...,...,...,...,...
7633,Zepto,D674,810439,2026-08-31,0.2828,404.0
7634,Zepto,D674,810518,2026-08-31,0.2160,216.0
7635,Zepto,D674,810673,2026-08-31,0.3360,24.0
7636,Zepto,D674,810674,2026-08-31,0.3360,24.0


In [78]:
final_df.to_csv('qcom_chain_forecast_aug2.csv')

In [276]:
# material_master_df[['parent_material_code', 'brand_code']].dtypes
material_master_df['parent_material_code'] = material_master_df['parent_material_code'].astype(int)
material_master_df.loc[material_master_df['parent_material_code'].isin([725930,731857]), 'brand_code'] = 'H&C_ALMND'

# offtake_df.drop(columns = ['brand_code'],inplace = True)


In [277]:
material_master_df[['parent_material_code', 'brand_code']].drop_duplicates().duplicated(subset=[ 'parent_material_code']).sum()

0

In [278]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    material_master_df[['parent_material_code', 'brand_code']].drop_duplicates(),
    left_on=['realigned_psku'],right_on = ['parent_material_code'],
    how='left'
)
assert len_before_merge == len(final_df)

In [279]:
final_df

,chain_name,depot,realigned_psku,month_date,vol_in_rum,forecast_quantity,parent_material_code,brand_code
0,Blinkit,D112,718288,2026-09-30,0.64200,107.0,718288,SAFF GOLD
1,Blinkit,D112,718312,2026-09-30,1.11000,1110.0,718312,PCNO(R)
2,Blinkit,D112,718322,2026-09-30,2.87500,575.0,718322,SAFF KO
3,Blinkit,D112,718328,2026-09-30,0.39060,434.0,718328,SAFF KOCO
4,Blinkit,D112,718330,2026-09-30,0.34500,69.0,718330,SAFF KOCO
...,...,...,...,...,...,...,...,...
6640,Zepto,D674,718850,2026-09-30,39.70000,397.0,718850,PADV-HRCR
6641,Zepto,D674,719085,2026-09-30,535.64000,1913.0,719085,PA_CN_HO
6642,Zepto,D674,721427,2026-09-30,1.53500,1535.0,721427,SAFF OATS
6643,Zepto,D674,727811,2026-09-30,16.29900,1811.0,727811,LIVON S-R


In [280]:
df_chk

,chain_name,depot,parent_material_code,month_date,vol_in_rum,forecast_quantity
0,Blinkit,D112,718288,2026-09-30,0.6420,107
1,Blinkit,D112,718312,2026-09-30,1.1100,1110
2,Blinkit,D112,718322,2026-09-30,2.8750,575
3,Blinkit,D112,718328,2026-09-30,0.3906,434
4,Blinkit,D112,718330,2026-09-30,0.3450,69
...,...,...,...,...,...,...
6493,Swiggy,D677,810673,2026-09-30,0.5040,36
6494,Swiggy,D677,810674,2026-09-30,0.5040,36
6495,Swiggy,D677,810738,2026-09-30,0.0000,0
6496,Swiggy,D677,811181,2026-09-30,0.0240,24


In [281]:
def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate



In [282]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


len_before_merge = len(final_df)

final_df = final_df.rename(columns={'material_group_code': 'brand_code'}).merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(final_df)


Credentials retrieved successfully for prod db.


In [284]:
#final_df['value'] = final_df['vol_in_rum']*final_df['qtr_ind_rate']/10**7
final_df[final_df['chain_name'] == 'Zepto'].groupby(['month_date'])['value'].sum()

month_date
2026-09-30    6.845372
Name: value, dtype: float64

In [285]:
final_df

,chain_name,depot,realigned_psku,month_date,vol_in_rum,forecast_quantity,parent_material_code,brand_code,qtr_ind_rate,value
0,Blinkit,D112,718288,2026-09-30,0.64200,107.0,718288,SAFF GOLD,138865.260689,0.008915
1,Blinkit,D112,718312,2026-09-30,1.11000,1110.0,718312,PCNO(R),349274.001420,0.038769
2,Blinkit,D112,718322,2026-09-30,2.87500,575.0,718322,SAFF KO,168827.536176,0.048538
3,Blinkit,D112,718328,2026-09-30,0.39060,434.0,718328,SAFF KOCO,123636.889888,0.004829
4,Blinkit,D112,718330,2026-09-30,0.34500,69.0,718330,SAFF KOCO,123636.889888,0.004265
...,...,...,...,...,...,...,...,...,...,...
6640,Zepto,D674,718850,2026-09-30,39.70000,397.0,718850,PADV-HRCR,569.875983,0.002262
6641,Zepto,D674,719085,2026-09-30,535.64000,1913.0,719085,PA_CN_HO,348.274000,0.018655
6642,Zepto,D674,721427,2026-09-30,1.53500,1535.0,721427,SAFF OATS,127515.619406,0.019574
6643,Zepto,D674,727811,2026-09-30,16.29900,1811.0,727811,LIVON S-R,1621.063608,0.002642


In [291]:
final_df.groupby(['chain_name'])['forecast_quantity'].sum()

chain_name
Blinkit    2269136.0
Swiggy      561865.0
Zepto       823226.0
Name: forecast_quantity, dtype: float64

In [297]:
final_df.isnull().sum()

chain_name              0
depot                   0
realigned_psku          0
month_date              0
vol_in_rum              0
forecast_quantity       0
parent_material_code    0
brand_code              0
qtr_ind_rate            0
value                   0
dtype: int64

In [298]:
final_df.to_csv('qcom_chain_forecast_sep2.csv')

### The end

In [71]:
primary = pd.read_csv('/data/aman_singh/acuuracy_check/QCOM Chain PSKU OTP Output/live_runs/QCOM Chain Depot PSKU Primary_live_runs_06_Jul_2026.csv')
primary

,Key2,Key,Chain,Depot,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,Offtake Chain depot PSKU Lag 3 Val,Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU Actuals Val,LY Offtake Chain depot PSKU Lag 1 Val,LY Offtake Chain depot PSKU Lag 2 Val,LY Offtake Chain depot PSKU Lag 3 Val,LY Offtake Chain depot PSKU Lead 1 Val,LY Offtake Chain depot PSKU Lead 2 Val,Calculated Primary Val
0,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212675,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212676,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212677,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212678,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-02-28,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [72]:
primary.columns

Index(['Key2', 'Key', 'Chain', 'Depot', 'PSKU', 'Brand', 'Index Rate',
       'Portfolio', 'Run Month', 'Month Date',
       ...
       'Offtake Chain depot PSKU Lag 3 Val',
       'Offtake Chain depot PSKU P3M Val',
       'LY Offtake Chain depot PSKU P3M Val',
       'LY Offtake Chain depot PSKU Actuals Val',
       'LY Offtake Chain depot PSKU Lag 1 Val',
       'LY Offtake Chain depot PSKU Lag 2 Val',
       'LY Offtake Chain depot PSKU Lag 3 Val',
       'LY Offtake Chain depot PSKU Lead 1 Val',
       'LY Offtake Chain depot PSKU Lead 2 Val', 'Calculated Primary Val'],
      dtype='object', length=117)

In [73]:
final_df.columns = ['Chain', 'Depot', 'PSKU','Month Date','Chain_primary_vol']
final_df['Depot'] = final_df['Depot'].str.lower()
final_df

,Chain,Depot,PSKU,Month Date,Chain_primary_vol
0,Blinkit,d112,718288,2026-08-31,0.0420
1,Blinkit,d112,718312,2026-08-31,1.9460
2,Blinkit,d112,718322,2026-08-31,3.5400
3,Blinkit,d112,718328,2026-08-31,0.2151
4,Blinkit,d112,718330,2026-08-31,0.0250
...,...,...,...,...,...
1245,Zepto,d674,810518,2026-08-31,0.3080
1246,Zepto,d674,810519,2026-08-31,0.1000
1247,Zepto,d674,810673,2026-08-31,0.5040
1248,Zepto,d674,810674,2026-08-31,0.3360


In [74]:
primary['Month Date'] = pd.to_datetime(primary['Month Date'])
primary = primary.merge(final_df, on = ['Chain', 'Depot', 'PSKU','Month Date'], how = 'left')
primary['Chain_primary_val'] = primary['Chain_primary_vol']*primary['Index Rate']/10**7
primary

,Key2,Key,Chain,Depot,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,LY Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU Actuals Val,LY Offtake Chain depot PSKU Lag 1 Val,LY Offtake Chain depot PSKU Lag 2 Val,LY Offtake Chain depot PSKU Lag 3 Val,LY Offtake Chain depot PSKU Lead 1 Val,LY Offtake Chain depot PSKU Lead 2 Val,Calculated Primary Val,Chain_primary_vol,Chain_primary_val
0,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
1,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
2,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
3,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
4,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212675,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
212676,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
212677,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
212678,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-02-28,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN


In [75]:
primary[primary['Month Date']=='2026-08-31']['Chain_primary_val'].sum()

41.958339936520986

In [76]:
primary.to_csv('cdp_c_forecast.csv')

In [66]:
x['Chain_primary_vol'].isnull().sum()

207123

In [67]:
primary

,Key2,Key,Chain,Depot,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,Offtake Chain depot PSKU Lag 3 Val,Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU Actuals Val,LY Offtake Chain depot PSKU Lag 1 Val,LY Offtake Chain depot PSKU Lag 2 Val,LY Offtake Chain depot PSKU Lag 3 Val,LY Offtake Chain depot PSKU Lead 1 Val,LY Offtake Chain depot PSKU Lead 2 Val,Calculated Primary Val
0,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212675,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212676,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212677,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212678,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-02-28,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# swiggy_unpivoted.groupby(['chain_name','facility_name','item_code','date'])['forecast_quantity'].sum().reset_index()

,chain_name,facility_name,item_code,date,forecast_quantity
0,Swiggy,AHM DELHIVERY,3,2026-07-31,192
1,Swiggy,AHM DELHIVERY,3,2026-08-31,768
2,Swiggy,AHM DELHIVERY,3,2026-09-30,576
3,Swiggy,AHM DELHIVERY,102,2026-07-31,60
4,Swiggy,AHM DELHIVERY,102,2026-08-31,160
...,...,...,...,...,...
29878,Swiggy,VIZ IM1,995855,2026-08-31,0
29879,Swiggy,VIZ IM1,995855,2026-09-30,0
29880,Swiggy,VIZ IM1,999977,2026-07-31,4
29881,Swiggy,VIZ IM1,999977,2026-08-31,5


In [ ]:
blinkit_unpivoted = blinkit_unpivoted[['chain_name','facility_name','item_code','date','forecast_quantity']]
swiggy_unpivoted = swiggy_unpivoted[['chain_name','facility_name','item_code','date','forecast_quantity']]


In [ ]:
chain_forecast_unpivoted = pd.concat([blinkit_unpivoted,swiggy_unpivoted])
chain_forecast_unpivoted

,chain_name,facility_name,item_code,date,forecast_quantity
0,Blinkit,Surat S1 - Feeder Warehouse,10171388,2026-07-31,856
1,Blinkit,Surat S1 - Feeder Warehouse,10232351,2026-07-31,18
2,Blinkit,Surat S1 - Feeder Warehouse,10029462,2026-07-31,12
3,Blinkit,Surat S1 - Feeder Warehouse,10015827,2026-07-31,209
4,Blinkit,Surat S1 - Feeder Warehouse,10116052,2026-07-31,65
...,...,...,...,...,...
29878,Swiggy,PUN DELHIVERY,990631,2026-09-30,22
29879,Swiggy,CHD ECOM,991861,2026-09-30,24
29880,Swiggy,CHN ECOM,995855,2026-09-30,0
29881,Swiggy,HYD IM1,998784,2026-09-30,0


In [ ]:
len_before_merge = len(chain_forecast_unpivoted)
chain_forecast_unpivoted['item_code'] = chain_forecast_unpivoted['item_code'].astype(str)
mapping['asin'] = mapping['asin'].astype(str)
temp = mapping[['platform_name','asin','EAN','PSKU','UOM','Vol per unit']].drop_duplicates()

temp = temp[temp['platform_name'].isin(['Blinkit', 'Swiggy', 'Zepto'])]
#temp['platform_name'].unique()
duplicates = temp[temp.duplicated(subset="asin", keep=False)]
duplicates



,platform_name,asin,EAN,PSKU,UOM,Vol per unit


In [ ]:
# # Keys to match rows on
# keys = ["platform_name", "asin", "EAN", "PSKU", "UOM", "Vol per unit"]

# # Build a small DataFrame with the rows to drop
# rows_to_drop = pd.DataFrame([
#     {
#         "platform_name": "Zepto",
#         "asin": "0523a4ba-32cf-4e59-abd8-0e4086859b39",
#         "EAN": "8901088205924",
#         "PSKU": "718729",
#         "UOM": "L",
#         "Vol per unit": 100.0,
#     },
#     {
#         "platform_name": "Zepto",
#         "asin": "197827dc-3184-4c57-a966-5461967bcb3a",
#         "EAN": "8901088884402",
#         "PSKU": "808485",
#         "UOM": "L",
#         "Vol per unit": 150.0,
#     },
#     {
#         "platform_name": "Zepto",
#         "asin": "82d8e93d-3d18-44b3-9904-3bcf521d0204",
#         "EAN": "8906051370753",
#         "PSKU": "807069",
#         "UOM": "L",
#         "Vol per unit": 150.0,
#     },
#     {
#         "platform_name": "Zepto",
#         "asin": "82d8e93d-3d18-44b3-9904-3bcf521d0204",
#         "EAN": "8901088075817",
#         "PSKU": "808262",
#         "UOM": "L",
#         "Vol per unit": 150.0,
#     },
#     {
#         "platform_name": "Swiggy",
#         "asin": "944906",
#         "EAN": "8901088150095",
#         "PSKU": "718976",
#         "UOM": "L",
#         "Vol per unit": 300.0,
#     },
# ])

# # Mark rows to drop via left-merge on keys
# _marked = temp.merge(
#     rows_to_drop.assign(_drop=1),
#     on=keys,
#     how="left",
#     validate="m:m"  # remove if unsure about duplicates
# )

# # Keep everything that was not marked to drop
# temp_clean = _marked[_marked["_drop"].isna()].drop(columns=["_drop"])
# temp_clean
# duplicates = temp_clean[temp_clean.duplicated(subset="asin", keep=False)]
# duplicates

In [ ]:
temp['PSKU'] = temp['PSKU'].astype(str)
temp['EAN'] = temp['EAN'].astype(str)
temp['UOM'] = temp['UOM'].astype(str)
len_before_merge = len(chain_forecast_unpivoted)
df_chk = chain_forecast_unpivoted.merge(temp,
                  left_on = ['item_code'], right_on = ['asin'], how = 'left')
assert(len_before_merge == len(df_chk))
df_chk['date'] = pd.to_datetime(df_chk['date'])

In [ ]:
df_chk

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
0,Blinkit,Surat S1 - Feeder Warehouse,10171388,2026-07-31,856,Blinkit,10171388,8901088213608,721427,TO,1000.0
1,Blinkit,Surat S1 - Feeder Warehouse,10232351,2026-07-31,18,Blinkit,10232351,8901088796804,810520,KL,1000.0
2,Blinkit,Surat S1 - Feeder Warehouse,10029462,2026-07-31,12,Blinkit,10029462,6001159111856,807033,L,125.0
3,Blinkit,Surat S1 - Feeder Warehouse,10015827,2026-07-31,209,Blinkit,10015827,8901088043953,718312,KL,1000.0
4,Blinkit,Surat S1 - Feeder Warehouse,10116052,2026-07-31,65,Blinkit,10116052,8901088205993,721133,L,300.0
...,...,...,...,...,...,...,...,...,...,...,...
50086,Swiggy,PUN DELHIVERY,990631,2026-09-30,22,NaN,NaN,NaN,NaN,NaN,NaN
50087,Swiggy,CHD ECOM,991861,2026-09-30,24,Swiggy,991861,8906027074531,729893,L,240.0
50088,Swiggy,CHN ECOM,995855,2026-09-30,0,NaN,NaN,NaN,NaN,NaN,NaN
50089,Swiggy,HYD IM1,998784,2026-09-30,0,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
duplicates.isnull().sum()

chain_name               0
facility_name            0
item_code                0
date                     0
forecast_quantity        0
platform_name        12111
asin                 12111
EAN                  12111
PSKU                 12111
UOM                  12111
Vol per unit         12111
dtype: int64

In [ ]:
df_chk = df_chk.dropna(subset = ['PSKU'])
df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False).sum()

474

In [ ]:
# duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
# duplicates.sort_values(by=['chain_name','facility_name','PSKU','date'])[:60]

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
20560,Swiggy,AHM DELHIVERY,554793,2026-07-31,0,Swiggy,554793,8901088886970,809042,KG,225.0
23718,Swiggy,AHM DELHIVERY,60103,2026-07-31,60,Swiggy,60103,8901088886970,809042,KG,225.0
30521,Swiggy,AHM DELHIVERY,554793,2026-08-31,0,Swiggy,554793,8901088886970,809042,KG,225.0
33679,Swiggy,AHM DELHIVERY,60103,2026-08-31,60,Swiggy,60103,8901088886970,809042,KG,225.0
40482,Swiggy,AHM DELHIVERY,554793,2026-09-30,0,Swiggy,554793,8901088886970,809042,KG,225.0
43640,Swiggy,AHM DELHIVERY,60103,2026-09-30,120,Swiggy,60103,8901088886970,809042,KG,225.0
21827,Swiggy,BLR DHL,819548,2026-07-31,192,Swiggy,819548,8901088171755,719162,TO,250.0
23859,Swiggy,BLR DHL,298412,2026-07-31,0,Swiggy,298412,8901088171755,719162,TO,250.0
31788,Swiggy,BLR DHL,819548,2026-08-31,192,Swiggy,819548,8901088171755,719162,TO,250.0
33820,Swiggy,BLR DHL,298412,2026-08-31,0,Swiggy,298412,8901088171755,719162,TO,250.0


In [ ]:
# duplicates.sort_values(by=['chain_name','facility_name','PSKU','date']).to_csv('duplicates_swiggy2.csv')

In [ ]:
# df_chk = df_chk.sort_values('forecast_quantity', ascending=False) \
#        .drop_duplicates(subset=['chain_name', 'facility_name','PSKU' , 'date'], keep='first')
# df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False).sum()

0

In [ ]:
df_chk.columns

Index(['chain_name', 'facility_name', 'item_code', 'date', 'forecast_quantity',
       'platform_name', 'asin', 'EAN', 'PSKU', 'UOM', 'Vol per unit'],
      dtype='object')

In [ ]:
df_chk['month_date'] = df_chk['date'] + pd.offsets.MonthEnd(0)

df_chk.rename(columns = {'item_code':'platform_code', 'EAN':'eancode', 'UOM':'uom_reporting',
                         'Vol per unit':'vol_per_unit'},inplace=True)
df_chk['vol_in_lit'] = df_chk['forecast_quantity']*df_chk['vol_per_unit']/1000
df_chk['vol_in_rum'] = df_chk.apply(
    lambda x: x['vol_in_lit'] / 1000 if x['uom_reporting'] in ['KL', 'TO'] else x['vol_in_lit'],
    axis=1
)

df_chk = df_chk.groupby(['chain_name','facility_name', 'PSKU','month_date'])[['vol_in_rum']].sum().reset_index()
df_chk['PSKU'] = df_chk['PSKU'].astype(int)
df_chk.rename(columns = {'PSKU':'parent_material_code'}, inplace = True)
df_chk

,chain_name,facility_name,parent_material_code,month_date,vol_in_rum,forecast_quantity
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-07-31,5.754,959
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-08-31,6.204,1034
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-09-30,6.222,1037
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-10-31,8.568,1428
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,718312,2026-07-31,0.501,501
...,...,...,...,...,...,...
37674,Swiggy,VIZ IM1,810685,2026-08-31,0.008,20
37675,Swiggy,VIZ IM1,810685,2026-09-30,0.008,20
37676,Swiggy,VIZ IM1,810738,2026-07-31,0.000,0
37677,Swiggy,VIZ IM1,810738,2026-08-31,0.000,0


In [ ]:
df_chk.duplicated(subset=['chain_name','facility_name','parent_material_code','month_date'], keep=False).sum()

0